In [235]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from xgboost import XGBRegressor
from sklearn import model_selection
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor

In [236]:
data_file = Path("..") / "data" / "data_processed" / "SUPERSTORE_DATASET_MODELAGEM.csv"
df = pd.read_csv(data_file, sep=",")

In [237]:
df.head()

,ORDER_DATE,SHIP_DATE,SHIP_MODE,CUSTOMER_ID,SEGMENT,COUNTRY,CITY,STATE,REGION,PRODUCT_ID,CATEGORY,SUB_CATEGORY,PRODUCT_NAME,PROFIT,NET_SALES,PROFIT_SCALED,NET_SALES_SCALED
0,2019-01-03,2019-01-07,STANDARD CLASS,DP-13000,CONSUMER,USA,HOUSTON,TX,CENTRAL,OFF-PA-10000174,OFFICE SUPPLIES,PAPER,"MESSAGE BOOK, WIREBOUND, FOUR 5 1/2"" X 4"" FORM...",5.5512,26.3168,-0.115131,-0.279599
1,2019-01-04,2019-01-08,STANDARD CLASS,PO-19195,HOME OFFICE,USA,NAPERVILLE,IL,CENTRAL,OFF-LA-10003223,OFFICE SUPPLIES,LABELS,AVERY 508,4.2717,28.2816,-0.120738,-0.279036
2,2019-01-05,2019-01-12,STANDARD CLASS,MB-18085,CONSUMER,USA,PHILADELPHIA,PA,EAST,OFF-AR-10003478,OFFICE SUPPLIES,ART,AVERY HI-LITER EVERBOLD PEN STYLE FLUORESCENT ...,4.8840,46.8864,-0.118055,-0.273708
3,2019-01-06,2019-01-07,FIRST CLASS,JO-15145,CORPORATE,USA,ATHENS,GA,SOUTH,OFF-AR-10002399,OFFICE SUPPLIES,ART,"DIXON PRANG WATERCOLOR PENCILS, 10-COLOR SET W...",5.2398,38.3400,-0.116495,-0.276156
4,2019-01-06,2019-01-08,SECOND CLASS,LS-17230,CONSUMER,USA,LOS ANGELES,CA,WEST,OFF-PA-10002005,OFFICE SUPPLIES,PAPER,XEROX 225,9.3312,58.3200,-0.098564,-0.270433


In [238]:
df_predict = df[['SHIP_MODE', 'PROFIT_SCALED', 'NET_SALES_SCALED', 'REGION']].copy()

In [239]:
df_predict = pd.get_dummies(
    df_predict,
    columns=['REGION'])

In [240]:
df_predict['SHIP_MODE'].value_counts()

SHIP_MODE
STANDARD CLASS    1631
SECOND CLASS       563
FIRST CLASS        438
SAME DAY           137
Name: count, dtype: int64

In [241]:
state_map = {
        'STANDARD CLASS': '0',
        'SECOND CLASS': '1',
        'FIRST CLASS': '2',
        'SAME DAY': '3'
}

df_predict['SHIP_MODE'] = df_predict['SHIP_MODE'].map(state_map)
df_predict['SHIP_MODE'] = df_predict['SHIP_MODE'].astype('Int64')

In [242]:
df_predict.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2769 entries, 0 to 2768
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   SHIP_MODE         2769 non-null   Int64  
 1   PROFIT_SCALED     2769 non-null   float64
 2   NET_SALES_SCALED  2769 non-null   float64
 3   REGION_CENTRAL    2769 non-null   bool   
 4   REGION_EAST       2769 non-null   bool   
 5   REGION_SOUTH      2769 non-null   bool   
 6   REGION_WEST       2769 non-null   bool   
dtypes: Int64(1), bool(4), float64(2)
memory usage: 78.5 KB


In [243]:
df_predict.head()

,SHIP_MODE,PROFIT_SCALED,NET_SALES_SCALED,REGION_CENTRAL,REGION_EAST,REGION_SOUTH,REGION_WEST
0,0,-0.115131,-0.279599,True,False,False,False
1,0,-0.120738,-0.279036,True,False,False,False
2,0,-0.118055,-0.273708,False,True,False,False
3,2,-0.116495,-0.276156,False,False,True,False
4,1,-0.098564,-0.270433,False,False,False,True


In [244]:
X_train, X_test, y_train, y_test = train_test_split(
    df_predict.drop('NET_SALES_SCALED', axis=1),
    df_predict['NET_SALES_SCALED'],
    test_size=0.2,
    random_state=0
)

In [245]:
models = {
    'XGBOOST': XGBRegressor(random_state=0),
    'REGRESSAO LINEAR': LinearRegression(),
    'REGRESSAO POLINOMIAL': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('model', LinearRegression())
    ]),
    'RANDOM FOREST': RandomForestRegressor(random_state=0),
}

kfold = model_selection.KFold(
    n_splits=10,
    shuffle=True,
    random_state=0
)

print('=' * 30)
print('RESULTADOS DA EXPLORACAO (REGRESSAO)')
print('=' * 30, '\n')

results = []

for name, model in models.items():
    scores = model_selection.cross_validate(
        model,
        X_train,
        y_train,
        scoring=['neg_mean_squared_error', 'neg_mean_absolute_error', 'r2'],
        cv=kfold,
        n_jobs=-1
    )

    mse = round(-scores['test_neg_mean_squared_error'].mean(), 4)
    mae = round(-scores['test_neg_mean_absolute_error'].mean(), 4)
    r2 = round(scores['test_r2'].mean(), 4)

    result = {
        'name': name,
        'mse': mse,
        'mae': mae,
        'r2': r2
    }
    results.append(result)

    print(
        'NOME:', name,
        '\n\n-MSE:', mse,
        '\n-MAE:', mae,
        '\n-R2:', r2,
        '\n\n',
        '-' * 30,
        '\n',
        sep=''
    )

results_df = pd.DataFrame(results).sort_values('r2', ascending=False).reset_index(drop=True)
display(results_df)

RESULTADOS DA EXPLORACAO (REGRESSAO)

NOME:XGBOOST

-MSE:0.0048
-MAE:0.0443
-R2:0.1467

------------------------------

NOME:REGRESSAO LINEAR

-MSE:0.0036
-MAE:0.0403
-R2:0.3533

------------------------------

NOME:REGRESSAO POLINOMIAL

-MSE:0.0037
-MAE:0.0403
-R2:0.3492

------------------------------

NOME:RANDOM FOREST

-MSE:0.0041
-MAE:0.0416
-R2:0.2766

------------------------------



,name,mse,mae,r2
0,REGRESSAO LINEAR,0.0036,0.0403,0.3533
1,REGRESSAO POLINOMIAL,0.0037,0.0403,0.3492
2,RANDOM FOREST,0.0041,0.0416,0.2766
3,XGBOOST,0.0048,0.0443,0.1467
